In [5]:
from dotenv import load_dotenv  # импортируем функцию для загрузки .env файла
load_dotenv()  # загружаем переменные окружения из .env (API ключи и т.д.)

from gitsource import GithubRepositoryDataReader, chunk_documents  # читалка GitHub репо + разбивка текста на чанки
from minsearch import Index  # минималистичный поисковый индекс для векторного/текстового поиска
from openai import OpenAI  # официальный клиент OpenAI API

from rag_helper_hw import RAGBase  # базовый класс с логикой RAG pipeline

QUERY = "How does the agentic loop keep calling the model until it stops?"  # тестовый вопрос для RAG
openai_client = OpenAI()  # инициализируем OpenAI клиент (подхватывает OPENAI_API_KEY из .env)

In [6]:
reader = GithubRepositoryDataReader(  # создаём ридер для чтения файлов из GitHub репозитория
    repo_owner="DataTalksClub",       # владелец репозитория (организация или юзер)
    repo_name="llm-zoomcamp",         # название репозитория
    commit_id="8c1834d",              # конкретный коммит — фиксируем версию, чтобы результаты были воспроизводимы
    allowed_extensions={"md"},        # читаем только .md файлы (Markdown документация)
    filename_filter=lambda path: "/lessons/" in path,  # фильтр: берём только файлы из папки /lessons/
)
files = reader.read()  # скачиваем/читаем файлы из репо по заданным параметрам

documents = [file.parse() for file in files]  # парсим каждый файл в словарь {'filename', 'content'}
documents[0]  # смотрим на первый документ — проверка что всё прочиталось корректно

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
print("Q1:", len(documents))


Q1: 72


In [8]:
def build_index(docs):
    index = Index(text_fields=["content"], keyword_fields=["filename"])
    index.fit(docs)
    return index

index = build_index(documents)
results = index.search(QUERY, num_results=5)
print("Q2:", results[0]["filename"])

Q2: 01-agentic-rag/lessons/14-agentic-loop.md


In [10]:
results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [11]:
rag = RAGBase(index=index, llm_client=openai_client)
res = rag.rag(QUERY)
print("Q3 input tokens:", res.input_tokens)
print(res.answer)

Q3 input tokens: 7111
It keeps calling the model in a `while True` loop, and after each response it checks whether there were any `function_call` items.

- If the model returns a function call, the code runs the tool, appends the tool output to `messages`, and loops again.
- If the model returns only a normal message and no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls in the model’s response**.


In [12]:
chunks = chunk_documents(documents, size=2000, step=1000)
print("Q4:", len(chunks))

Q4: 295


In [13]:
chunk_index = build_index(chunks)
rag_chunked = RAGBase(index=chunk_index, llm_client=openai_client)
res_chunked = rag_chunked.rag(QUERY)
print("Q5 input tokens (chunked):", res_chunked.input_tokens)
print(f"Reduction: ~{res.input_tokens / res_chunked.input_tokens:.1f}x fewer")

Q5 input tokens (chunked): 2294
Reduction: ~3.1x fewer


In [14]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> list[dict]:
    """Search the course lessons for chunks matching the given query."""
    return chunk_index.search(query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)  # схема выводится из type hint + docstring

instructions = (
    "You're a course teaching assistant. Answer the student's question "
    "using the search tool. Make multiple searches with different keywords "
    "before answering."
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [15]:
search_calls = sum(
    1 for m in result.all_messages
    if getattr(m, "type", None) == "function_call" and getattr(m, "name", None) == "search"
)
print("Q6 search calls:", search_calls)

Q6 search calls: 4
